In [4]:
!pip install -q transformers torch gradio evaluate rouge_score sentencepiece

In [5]:
import gradio as gr
import torch
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


# ============================================================
# 1. LOAD SUMMARIZATION MODEL
# ============================================================

print("Loading summarization model...")

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully!")


# ============================================================
# 2. SUMMARIZATION FUNCTION
# ============================================================

def summarize_text(input_text):

    if not input_text or not input_text.strip():
        return "Please enter some text to summarize."

    # Convert input text into tokens
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    # Generate summary
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=45,
            min_length=15,
            do_sample=False
        )

    # Convert tokens back to text
    summary = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return summary


# ============================================================
# 3. CREATE GRADIO APPLICATION
# ============================================================

demo = gr.Interface(
    fn=summarize_text,

    inputs=gr.Textbox(
        lines=8,
        label="Enter text to summarize",
        placeholder="Enter a paragraph or article here..."
    ),

    outputs=gr.Textbox(
        label="Generated Summary"
    ),

    title="GenAI Text Summarizer",

    description=(
        "A Generative AI text summarization application "
        "using BART and Gradio."
    )
)


# ============================================================
# 4. LAUNCH APPLICATION
# ============================================================

demo.launch(share=True)

Loading summarization model...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c23a40aad6af924f0d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
import evaluate

# Load ROUGE metric
rouge = evaluate.load("rouge")

generated_summaries = [
    "AI models generate new content such as text and images."
]

reference_summaries = [
    "Generative AI models are capable of producing new content including text and images."
]

scores = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries
)

print("ROUGE Evaluation Scores:")

for metric, score in scores.items():
    print(f"{metric}: {score:.4f}")

ROUGE Evaluation Scores:
rouge1: 0.6087
rouge2: 0.3810
rougeL: 0.6087
rougeLsum: 0.6087
